# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [ ]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from bs4 import BeautifulSoup
import requests
from urllib.parse import urljoin
from openai import OpenAI
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [ ]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
gemini_api_key = os.getenv('GEMINI_API_KEY')
gemmini_model = os.getenv('GEMINI_MODEL')
GEMINI_BASE_URL = os.getenv('GEMINI_BASE_URL')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL_LLAMA = os.getenv('LAMMA_MODEL')
MODEL_GEMMA = os.getenv('GEMMA_MODEL_LAMMA')
openai = OpenAI()

OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
# gemini = OpenAI(base_url="https://gemini.api.openai.com/v1", api_key=gemini_api_key)
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=gemini_api_key)

In [ ]:
# A class to represent a Webpage

# Some websites need you to use proper headers when fetching them:

class Website:
    def __init__(self, url):
        self.url = url
        self.title = "No title found"
        self.text = ""
        self.soup = None  # Inicjalizujemy jako None
        
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        }
        try:
            session = requests.Session()
            # first request to get any cookies
            response = session.get(url, headers=headers, allow_redirects=True, timeout=10, verify=False)
            # checking result (code 200)
            response.raise_for_status() 
            
            self.body = response.content
            self.soup = BeautifulSoup(self.body, 'html.parser')
            
            if self.soup.title:
                self.title = self.soup.title.string
            if self.soup.body:
                self.text = self.soup.body.get_text(separator="\n", strip=True)
        except requests.exceptions.SSLError:
            print(f"SSL Error for {url}. Try updating your certificates or use verify=False.")        
        except Exception as e:
            print(f"Website error for {url}: {e}")

    def get_links(self):
        if not self.soup:
            return []
        
        candidates = []
        # key words to look for in links
        keywords = ["repertuar", "program", "bilety", "schedule", "kup"]
        # key worsds to avoid in links
        forbidden = ["screeningid", "eventid", "cinemaid", "article", "promocje"]
        
        seen_urls = set()
        for a in self.soup.find_all('a', href=True):
            href = urljoin(self.url, a['href'])
            text = a.get_text(strip=True).lower()
            href_lower = href.lower()         
            # 1. if link points to the same page, skip it
            if href.rstrip('/') == self.url.rstrip('/'):
                continue
            # 2. skip links with forbidden words
            if any(f in href_lower for f in forbidden):
                continue
            # 3. accept links with keywords in text or href
            if any(kw in text or kw in href_lower for kw in keywords):
                if href not in seen_urls:
                    candidates.append({
                        "text": a.get_text(strip=True) or "Link",
                        "url": href
                    })
                    seen_urls.add(href)
                    
        return candidates[:25] # Limit to first 25 links
    
    def get_clean_content(self):
        try:
            for element in self.soup(["script", "style", "head", "footer", "noscript", "svg", "form"]):
                element.decompose()
            allowed_attrs = ["href", "title", "class"]
            for tag in self.soup.find_all(True):
                tag.attrs = {name: value for name, value in tag.attrs.items() if name in allowed_attrs}
            raw_text = self.soup.get_text(separator=' | ', strip=True)
            # return ' '.join(raw_text.split())
            return raw_text
            # return self.soup.prettify()
        except:
            return b""  # return empty bytes on error 
    def get_content(self):
        try:
            return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"
        except:
            return "" 

In [ ]:
class CinemaAI:
    def __init__(self, client, model_name):
        self.client = client
        self.model = model_name

    def _call_ai(self, system_prompt, user_prompt):
        # ... 
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                response_format={"type": "json_object"},
                temperature=0
            )
            return json.loads(response.choices[0].message.content)
        except Exception as e:
            print(f"  > AI Error: {e}")
            return None

    def get_repertuar_link(self, base_url, links):
        """Logika HYBRYDOWA: Najpierw Python, potem AI"""
        
        # 1. SZYBKI FILTR PYTHON (Heurystyka)
        # Normalizujemy URL i szukamy słowa 'repertuar'
        perfect_matches = list(set([l['url'].rstrip('/') for l in links if "repertuar" in l['url'].lower()]))
        
        # Jeśli mamy jeden idealny link, bierzemy go bez pytania AI
        if len(perfect_matches) == 1:
            return perfect_matches[0]
        
        # Jeśli najkrótszy z idealnych wygląda na główny (np. /repertuar)
        if perfect_matches:
            shortest = min(perfect_matches, key=len)
            if shortest.endswith('/repertuar'):
                return shortest

        # 2. JEŚLI NIEPEWNOŚĆ -> PYTAMY AI
        print(f"  > Python unsure for {base_url}. Asking AI ({self.model})...")
        
        system_prompt = f"""You are a cinema expert. Pick the best link to the movie schedule.
Base URL: {base_url}
Return ONLY JSON: {{"repertuar_link": "URL"}}
Rules: Use EXACT URL from the list. Favor 'bilety24.pl' if no local repertoire exists."""
        
        # Przekazujemy przefiltrowane linki (np. tylko te z tekstem 'bilety', 'kup' itp.)
        user_prompt = f"List: {json.dumps(links[:20])}"
        
        decision = self._call_ai(system_prompt, user_prompt)
        
        # Weryfikacja halucynacji (czy wynik jest na liście)
        suggested = decision.get("repertuar_link") if decision else None
        valid_urls = [l['url'] for l in links]
        
        if suggested in valid_urls:
            return suggested
        return None
    
    def extract_movies(self, page_text):
        """Metoda do wyciągania filmów i godzin"""
        system_prompt = """Extract movies and showtimes for today. 
Return ONLY JSON format: {"movies": [{"title": "Title", "showtimes": ["HH:MM"]}]}"""
        
        # Ograniczamy tekst do 5000 znaków, by nie przekroczyć limitów mniejszych modeli
        return self._call_ai(system_prompt, page_text[:5000])

    def get_movie_description(self, movie_title, context=""):
        """Generuje opis na podstawie strony lub wiedzy ogólnej"""
        system_prompt = f"""Write a very short (1-2 sentences) summary of '{movie_title}' in Polish. 
If the context is empty, use your general knowledge.
Return ONLY JSON: {{"summary": "opis"}}"""
        
        user_prompt = f"Context: {context[:2000]}"
        return self._call_ai(system_prompt, user_prompt)
    
    def find_specific_movie(self, page_text, target_title):
        """
        Szuka konkretnego tytułu w tekście strony i zwraca jego godziny oraz opis.
        """
        system_prompt = f"""You are a cinema assistant. 
Search the provided text specifically for the movie: "{target_title}".
EXAMPLE:
Input: "Wicked | 12+ | 160 min | 17:00 | 20:00"
Output: {{"title": "Wicked", "showtimes": ["17:00", "20:00"]}}
RULE: Do not confuse the viewer's age (e.g. 15+, 12) with the screening time (format HH:MM).
Return ONLY JSON format: 
{{
  "found": true/false,
  "title": "Exact title found",
  "showtimes": ["HH:MM"],
  "description": "Short 1-2 sentence summary in Polish"
}}
If the movie is not in the text, return {{"found": false}}."""
        
        # Przekazujemy tekst strony jako kontekst
        user_prompt = f"Text to analyze: {page_text[:25000]}"
        
        return self._call_ai(system_prompt, user_prompt)

In [ ]:
url = "https://kinoteka.pl/repertuar"
website = Website(url)
links = website.get_links()
print(len(website.text))
print(website.text)


## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [ ]:
cinemas = [
     "https://kinoteka.pl/",
     "https://kinoluna.waw.pl/",
     "https://www.kinomuranow.pl/"
 ]

szukany_film = "Father Mother Sister Brother"
results = {}

ai_ollama = CinemaAI(ollama, MODEL_LLAMA)
# ai_gemini = CinemaAI(gemini, gemmini_model)
print("--- CINEMA REPERTOIRE SCANNER ---")
for url in cinemas:
    print(f"Processing: {url}...")
    site = Website(url)
    links = site.get_links()
    # Hybrydowe wyszukiwanie linku (Python -> AI)
    repertuar_url = ai_ollama.get_repertuar_link(url, links)
    results[site.title] = repertuar_url
    
    if repertuar_url:
        print(f"SUKCES: {repertuar_url}")
        rep_site = Website(repertuar_url)
        # Ensure we always work with a str (Website.get_clean_content may return bytes on error)
        raw_content = rep_site.get_clean_content()
        if isinstance(raw_content, bytes):
            text_site = raw_content.decode('utf-8', errors='ignore')[:8000]
        else:
            text_site = str(raw_content)[:8000]
        # collecting movies
        # movies_data = ai_ollama.extract_movies(rep_site.text)
        # looking for specific movie
        print(f"DEBUG: Czy tytuł jest w tekście strony? {'Tak' if 'Father' in text_site else 'Nie'}")
        print(f"DEBUG: Długość tekstu strony: {len(text_site)} znaków")
        movie_info = ai_ollama.find_specific_movie(text_site, szukany_film)
        if movie_info and movie_info.get("found"):
            print(f"Znalazłem w {url}:")
            print(f" - Film: {movie_info['title']}")
            print(f" - Godziny: {', '.join(movie_info['showtimes'])}")
            print(f" - Opis: {movie_info['description']}\n")
        else:
            print(f"Filmu '{szukany_film}' nie ma dziś w {url}.\n")
    else:
        print(f"PORAŻKA: Nie znaleziono repertuaru dla {url}")
    # print(f"Result: {repertuar}\n")

# print("--- FINAL SUMMARY ---")
# print(json.dumps(results, indent=4))

In [ ]:
website = Website("https://kinoteka.pl/repertuar")
print(website.get_clean_content().split())

In [ ]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
gemini_api_key = os.getenv('GEMINI_API_KEY')
gemmini_model = os.getenv('GEMINI_MODEL')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
# MODEL = 'gpt-5-nano'
openai = OpenAI()

OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
gemini = OpenAI(base_url="https://gemini.api.openai.com/v1", api_key=gemini_api_key)
MODEL = "llama3.2"

link_system_prompt = """
You are provided with a list of links to cinemas webpage.
You are able to find the most relevant link to "Repertuar" page about the movies being shown in the cinema,
such as links to a Repertuar page, or a Movies page.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "repertuar", "url": "https://full.url/goes/here/repertuar"},
    ]
}
"""

def get_links_user_prompt(cinemas_website_url):
    user_prompt = f"""
Here is the list of links to the cinemas website -
Please decide which of these are relevant web links for repertuar page about the movies being shown in the cinema, 
respond with the full https URL in JSON format.
Include only relevant link.

Links (some might be relative links):

"""
    for url in cinemas_website_url:
        links += fetch_website_links(url)

    user_prompt += "\n".join(links)
    return user_prompt


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>